Entrenamiento usando el modelo *mistralai/Mistral-7B-Instruct-v0.3* dándole un enfoque discriminativo.

In [1]:
!pip install -U bitsandbytes trl torchao accelerate transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 61.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.8/925.8 kB 62.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 131.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 158.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 52.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 49.7 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: transformers
    Fou

In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    BitsAndBytesConfig,
    Trainer,
    EarlyStoppingCallback,
    AutoModelForCausalLM,
    pipeline
)
from trl import SFTTrainer, SFTConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from tqdm import tqdm
import bitsandbytes as bnb
from sklearn.metrics import classification_report, f1_score, accuracy_score, confusion_matrix

In [3]:
model_checkpoint = "mistralai/Mistral-7B-Instruct-v0.3"
n_labels = 2

# Entrenamiento Train/Valid/Test

## Fase de entrenamiento

1. Configuración de cuantización a 4 bits

In [4]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 # Acelera los cálculos matemáticos
)

print("Cargando tokenizador y modelo base")

Cargando tokenizador y modelo base


2. Tokenizador

In [5]:
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/141k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  587kB            

tokenizer.model: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

3. Inicialización del Modelo Base Cuantizado (Con la cabeza matemática discriminativa)

In [6]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint,
    num_labels=n_labels,
    quantization_config=bnb_config,
    device_map="auto"
)
model.config.pad_token_id = tokenizer.pad_token_id

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[transformers] MistralForSequenceClassification LOAD REPORT from: mistralai/Mistral-7B-Instruct-v0.3
Key            | Status     | 
---------------+------------+-
lm_head.weight | UNEXPECTED | 
score.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


4. Buscador dinámico de capas para LoRA

In [7]:
def find_all_linear_names(model):
    cls = bnb.nn.Linear4bit
    lora_module_names = set()
    for name, module in model.named_modules():
        if isinstance(module, cls):
            names = name.split('.')
            lora_module_names.add(names[0] if len(names) == 1 else names[-1])
    if 'lm_head' in lora_module_names:
        lora_module_names.remove('lm_head')
    if 'score' in lora_module_names: # Excluimos la cabeza de clasificación de la cuantización normal
        lora_module_names.remove('score')
    return list(lora_module_names)

modules = find_all_linear_names(model)
print(f"Módulos objetivo encontrados para LoRA: {modules}")

Módulos objetivo encontrados para LoRA: ['k_proj', 'up_proj', 'down_proj', 'gate_proj', 'q_proj', 'o_proj', 'v_proj']


5. Configuración de LoRA

In [8]:
model = prepare_model_for_kbit_training(model)
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=modules,
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_CLS",      # Tarea: Clasificación de Secuencias (no CAUSAL_LM)
    modules_to_save=["score"] # IMPORTANTE: Descongelar la capa final matemática
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 41,951,232 || all params: 7,155,765,248 || trainable%: 0.5863


6. Preparación del dataset y partición

In [9]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [10]:
# ================================
# 1. Carga y preparación de datos (Multietiqueta)
# ================================
import pandas as pd
from sklearn.model_selection import train_test_split

# --- PARAMETRIZACIÓN DE LA ETIQUETA ---
# Descomenta SOLO UNA etiqueta para cada ronda de entrenamiento

#ETIQUETA_OBJETIVO = "IDEOLOGICAL-INEQUALITY"
#ETIQUETA_OBJETIVO = "STEREOTYPING-DOMINANCE"
#ETIQUETA_OBJETIVO = "OBJECTIFICATION"
#ETIQUETA_OBJETIVO = "SEXUAL-VIOLENCE"
ETIQUETA_OBJETIVO = "MISOGYNY-NON-SEXUAL-VIOLENCE"

print(f"--- Preparando datos para la etiqueta: {ETIQUETA_OBJETIVO} ---")

# --- RUTAS TAREA 3.3 ---
RUTA_TRAIN = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Task3.3/EXIST2025_train_3_3.csv"
RUTA_TEST = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Task3.3/EXIST2025_test_3_3.csv"

print("Cargando el dataset maestro de entrenamiento y test...")
df_train_master = pd.read_csv(RUTA_TRAIN)
test_df = pd.read_csv(RUTA_TEST)

# --- ADAPTACIÓN PARA HUGGING FACE ---
# Renombramos la columna objetivo a 'label'
df_train_master = df_train_master.rename(columns={ETIQUETA_OBJETIVO: "label"})
test_df = test_df.rename(columns={ETIQUETA_OBJETIVO: "label"})

df_train_master["label"] = df_train_master["label"].astype(int)
test_df["label"] = test_df["label"].astype(int)

# --- DIVISIÓN TRAIN/VALIDATION ---
train_df, val_df = train_test_split(
    df_train_master,
    test_size=0.10,
    stratify=df_train_master["label"],
    random_state=42
)

print("\nDistribución fichero de entrenamiento (Train):")
print(train_df['label'].value_counts())

print("\nDistribución fichero de validación (Valid):")
print(val_df['label'].value_counts())

print("\nDistribución fichero de test (estático):")
print(test_df['label'].value_counts())

--- Preparando datos para la etiqueta: MISOGYNY-NON-SEXUAL-VIOLENCE ---
Cargando el dataset maestro de entrenamiento y test...

Distribución fichero de entrenamiento (Train):
label
0    739
1    135
Name: count, dtype: int64

Distribución fichero de validación (Valid):
label
0    83
1    15
Name: count, dtype: int64

Distribución fichero de test (estático):
label
0    189
1     41
Name: count, dtype: int64


In [11]:
def tokenize_data(example):
    # Asumimos que tu columna con el texto de los vídeos se llama "text"
    return tokenizer(example["text"], padding="max_length", truncation=True, max_length=128)

train_dataset = Dataset.from_pandas(train_df)
valid_dataset = Dataset.from_pandas(val_df)

train_dataset.reset_format()
valid_dataset.reset_format()

columns_train = train_dataset.column_names
columns_valid = valid_dataset.column_names
columna_etiqueta = "label" if "label" in columns_train else "labels"

if columna_etiqueta in columns_train: columns_train.remove(columna_etiqueta)
if columna_etiqueta in columns_valid: columns_valid.remove(columna_etiqueta)

encoded_train_dataset = train_dataset.map(tokenize_data, batched=True, remove_columns=columns_train)
encoded_valid_dataset = valid_dataset.map(tokenize_data, batched=True, remove_columns=columns_valid)

Map:   0%|          | 0/874 [00:00<?, ? examples/s]

Map:   0%|          | 0/98 [00:00<?, ? examples/s]

7. Hiperparámetros del entrenamiento

In [12]:
# Ruta dinámica basada en la etiqueta para no sobrescribir modelos
OUTPUT_DIR_MISTRAL = f"/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Task3.3/Research/Mistral/{ETIQUETA_OBJETIVO}"
print(f"Los pesos de QLoRA se guardarán en: {OUTPUT_DIR_MISTRAL}")

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR_MISTRAL,
    num_train_epochs=5,
    learning_rate=2e-4,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=8,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    weight_decay=0.01,
    fp16=False,
    bf16=True,
    optim="paged_adamw_8bit",
    report_to="none",
    logging_strategy='epoch'
)

Los pesos de QLoRA se guardarán en: /content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Task3.3/Research/Mistral/MISOGYNY-NON-SEXUAL-VIOLENCE


8. Entrenamiento usando el trainer clásico (porque es el discriminativo)

In [13]:
import sklearn as sk
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score, f1_score


In [14]:
# Función para realizar distintas métricas en ejecución

def compute_metrics(eval_pred):

  ##############
  ## predictions son logits, que son tuplas de la forma [valor1, valor2]
  ## Por ejemplo [-1.5606991,  1.6122842] significa que ha predicho eso para un documento
  ## Eso es lo que pasa a la última capa del transformer (softmax si es binario)
  ## Por eso se utiliza el índice del valor máximo de la tupla, para decir que esa es la clase que predice

  ## label_ids = [0, 1, 1, 0, 1]  # Etiquetas reales
  ## predictions = [
  ##  [0.8, 0.2],  # Predicciones para la primera instancia
  ##  [0.3, 0.7],  # Predicciones para la segunda instancia
  ##  [0.1, 0.9],  # Predicciones para la tercera instancia
  ##  [0.9, 0.1],  # Predicciones para la cuarta instancia
  ##  [0.4, 0.6],  # Predicciones para la quinta instancia
  ##           ]

  ##############

  labels = eval_pred.label_ids
  preds = eval_pred.predictions.argmax(-1)

  # Compute precision, recall, F1-score, and support
  precision, recall, f1, _ = sk.metrics.precision_recall_fscore_support(labels, preds, average="macro")

  # Calculate F1-score for the minority class (label = 1)
  f1_minoritaria= f1_score(labels, preds, pos_label=1)

  # Calculate F1-score for the majority class (label = 0)
  f1_mayoritaria = f1_score(labels, preds, pos_label=0)

  # Calculate accuracy
  acc = sk.metrics.accuracy_score(labels, preds)

  # Calculate Area Under the Curve (AUC)
  AUC = roc_auc_score(labels, preds)

  # Calculate Precision-Recall Area Under the Curve (AUC)
  PREC_REC = average_precision_score(labels, preds)

  return {
      'accuracy': acc,
      'f1': f1,
      'precision': precision,
      'recall': recall,
      'AUC': AUC,
      'f1_minoritaria': f1_minoritaria,
      'f1_mayoritaria': f1_mayoritaria,
      'PREC_REC': PREC_REC
  }

In [15]:
trainer = Trainer(
    model=model,
    args=training_args,
    compute_metrics=compute_metrics, # ¡TU FUNCIÓN DE MÉTRICAS ORIGINAL!
    train_dataset=encoded_train_dataset,
    eval_dataset=encoded_valid_dataset,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print(f"🚀 Iniciando entrenamiento de Mistral-7B para la etiqueta: {ETIQUETA_OBJETIVO}...")
trainer.train()

# ESTA ES LA MAGIA: Guardamos el mejor modelo directamente en la carpeta raíz
print(f"💾 Guardando el MEJOR modelo en la ruta principal...")
trainer.save_model(OUTPUT_DIR_MISTRAL)
tokenizer.save_pretrained(OUTPUT_DIR_MISTRAL)

print("✅ ¡Entrenamiento y guardado completados!")

🚀 Iniciando entrenamiento de Mistral-7B para la etiqueta: MISOGYNY-NON-SEXUAL-VIOLENCE...


[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!
/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall,Auc,F1 Minoritaria,F1 Mayoritaria,Prec Rec
1,10.559517,3.714045,0.163265,0.145833,0.577320,0.506024,0.506024,0.267857,0.023810,0.154639
2,3.468737,0.585299,0.755102,0.430233,0.415730,0.445783,0.445783,0.000000,0.860465,0.153061
3,1.284035,0.780229,0.816327,0.449438,0.421053,0.481928,0.481928,0.000000,0.898876,0.153061
4,0.397441,2.550143,0.816327,0.449438,0.421053,0.481928,0.481928,0.000000,0.898876,0.153061
5,0.031210,2.506370,0.826531,0.504609,0.550532,0.515261,0.515261,0.105263,0.903955,0.159524


/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/pyt

💾 Guardando el MEJOR modelo en la ruta principal...
✅ ¡Entrenamiento y guardado completados!


## Resultados contra fichero de Test

### 1. Configuración

In [16]:
MODEL_ID_BASE = "mistralai/Mistral-7B-Instruct-v0.3"

# Apuntamos DIRECTAMENTE a la carpeta principal que definimos arriba
DIR_MODELO_QLORA = f"/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Task3.3/Research/Mistral/{ETIQUETA_OBJETIVO}"

# Rutas de Test (Actualizadas a la Tarea 3.3)
CSV_TEST_TEXT = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Task3.3/EXIST2025_test_3_3.csv"

# Guardamos las predicciones con el nombre de la etiqueta para no pisarlas
CSV_SALIDA = f"/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Task3.3/Research/Mistral/predicciones_mistral_{ETIQUETA_OBJETIVO}.csv"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Configuración lista para evaluar: {ETIQUETA_OBJETIVO}")

Configuración lista para evaluar: MISOGYNY-NON-SEXUAL-VIOLENCE


### 2. Carga de datos de test

In [17]:
print("Cargando el dataset de test fijo...")
test_df = pd.read_csv(CSV_TEST_TEXT)

if "Unnamed: 0" in test_df.columns:
    test_df = test_df.drop(columns=["Unnamed: 0"])
test_df = test_df.rename(columns={ETIQUETA_OBJETIVO: "label"})
test_df["label"] = test_df["label"].astype(int)

# Guardamos la verdad absoluta
y_true = test_df["label"].tolist()

# Función para generar el prompt de evaluación (sin la respuesta)
def generate_test_prompt(data_point):
    return f"""
Clasifica el siguiente texto extraído de un vídeo de redes sociales en una de estas dos categorías: 'Misógino' o 'No misógino'. Devuelve ÚNICAMENTE la etiqueta correspondiente.
text: {data_point["text"]}
label: """.strip()

Cargando el dataset de test fijo...


### 3. Carga del modelo y tokenizador (PEFT/QLoRA)

In [18]:
print(f"Cargando Tokenizador de {MODEL_ID_BASE}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID_BASE)
tokenizer.padding_side = "right"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Cargando Modelo Base (SequenceClassification) en 16-bits...")
base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID_BASE,
    num_labels=2,            # Fundamental: Le decimos que es para 2 clases
    device_map="auto",
    torch_dtype=torch.bfloat16
)
base_model.config.pad_token_id = tokenizer.pad_token_id

print(f"Aplicando pesos QLoRA (incluyendo la capa de 'score') desde {DIR_MODELO_QLORA}...")
# Esto cargará la cabeza clasificadora gracias a tu "modules_to_save=['score']"
model = PeftModel.from_pretrained(base_model, DIR_MODELO_QLORA)
model.eval()

Cargando Tokenizador de mistralai/Mistral-7B-Instruct-v0.3...
Cargando Modelo Base (SequenceClassification) en 16-bits...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[transformers] MistralForSequenceClassification LOAD REPORT from: mistralai/Mistral-7B-Instruct-v0.3
Key            | Status     | 
---------------+------------+-
lm_head.weight | UNEXPECTED | 
score.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Aplicando pesos QLoRA (incluyendo la capa de 'score') desde /content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Task3.3/Research/Mistral/MISOGYNY-NON-SEXUAL-VIOLENCE...


PeftModelForSequenceClassification(
  (base_model): LoraModel(
    (model): MistralForSequenceClassification(
      (model): MistralModel(
        (embed_tokens): Embedding(32768, 4096)
        (layers): ModuleList(
          (0-31): 32 x MistralDecoderLayer(
            (self_attn): MistralAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )


### 4. Inferencia y parseo de las respuestas

In [19]:
print("Iniciando inferencia (extracción de probabilidades)...")
predicciones_csv = []
y_pred = []

for index, row in tqdm(test_df.iterrows(), total=len(test_df)):
    # Usamos el texto puro, tal cual lo hiciste en el entrenamiento
    texto = str(row["text"])
    id_video = row["id_EXIST"]

    # Tokenizamos
    inputs = tokenizer(texto, return_tensors="pt", padding="max_length", truncation=True, max_length=128).to(device)

    with torch.no_grad():
        outputs = model(**inputs)

        # Obtenemos los logits y aplicamos Softmax para sacar porcentajes (0 a 1)
        logits = outputs.logits
        probabilidades = F.softmax(logits, dim=-1)[0]

        # probabilidad de la clase 1 (Misógino)
        prob_misogino = probabilidades[1].item()

    # Decisión binaria (Umbral estándar > 0.5)
    pred_binaria = 1 if prob_misogino > 0.5 else 0
    y_pred.append(pred_binaria)

    # Guardamos los datos para el Ensemble
    predicciones_csv.append({
        "id_EXIST": id_video,
        "prob_misogino": prob_misogino,
        "prediccion_binaria": pred_binaria,
        "label_real": row["label"]
    })

Iniciando inferencia (extracción de probabilidades)...


100%|██████████| 230/230 [00:22<00:00, 10.14it/s]


### 5. Evaluación y métricas

In [20]:
print("\n" + "="*50)
print(f"🏆 RESULTADOS REALES EN TEST FIJO: Mistral (Sequence Classification) para la etiqueta {ETIQUETA_OBJETIVO}")
print("="*50)

f1 = f1_score(y_true, y_pred, average="macro")
acc = accuracy_score(y_true, y_pred)

print(f"\nF1-Score (Macro): {f1:.4f}")
print(f"Accuracy: {acc:.4f}")

print("\nMatriz de Confusión:")
print(confusion_matrix(y_true, y_pred))

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=["No Misógino", "Misógino"]))


🏆 RESULTADOS REALES EN TEST FIJO: Mistral (Sequence Classification) para la etiqueta MISOGYNY-NON-SEXUAL-VIOLENCE

F1-Score (Macro): 0.4861
Accuracy: 0.8043

Matriz de Confusión:
[[183   6]
 [ 39   2]]

Classification Report:
              precision    recall  f1-score   support

 No Misógino       0.82      0.97      0.89       189
    Misógino       0.25      0.05      0.08        41

    accuracy                           0.80       230
   macro avg       0.54      0.51      0.49       230
weighted avg       0.72      0.80      0.75       230



### 6. Guardar CSV para ensemble

In [21]:
df_salida = pd.DataFrame(predicciones_csv)
df_salida.to_csv(CSV_SALIDA, index=False)
print(f"\n✅ ¡CSV para el Ensemble guardado correctamente en: {CSV_SALIDA}!")


✅ ¡CSV para el Ensemble guardado correctamente en: /content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Task3.3/Research/Mistral/predicciones_mistral_MISOGYNY-NON-SEXUAL-VIOLENCE.csv!
